In [1]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt
import seaborn as sns

In [2]:
FOLDER = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
AFOROS_PATH = r"Insumos IMEPLAN\Aforos\Consolidado de Aforos 120826.xlsx"

### Read aforos

In [24]:
aforos = pd.read_excel(os.path.join(FOLDER, AFOROS_PATH))

# Keep only aforos of 2023
aforos2023 = aforos[aforos['Año']==2023].copy()
print(f"Los aforos de 2023 representan el {(len(aforos2023)/len(aforos))*100:.2f}% de los aforos")

Los aforos de 2023 representan el 74.70% de los aforos


### Agrupar por periodo

| Period Code | Period Name | Start Hour | End Hour |
|--------------|----------|-------|-------|
| EM | Early Morning    | 00:00 | 03:59 |
| AP | AM Peak          | 04:00 | 08:59 |
| LM | Late Morning     | 9:00  | 11:59 |
| EA | Early Afternoon  | 12:00 | 17:59 |
| PP | PM Peak          | 18:00 | 20:59 |
| LN | Late Night       | 21:00 | 23:59 |


In [19]:
def assign_period(hour):
    if 0 <= hour <= 3:
        return "EM"
    elif 4 <= hour <= 8:
        return "AP"
    elif 9 <= hour <= 11:
        return "LM"
    elif 12 <= hour <= 17:
        return "EA"
    elif 18 <= hour <= 20:
        return "PP"
    else:
        return "LN"

In [25]:
# quitar espacios por si viene como "06:00 - 06:15"
aforos2023["HORARIO"] = aforos2023["HORARIO"].astype(str).str.replace(" ", "", regex=False)

# sacar la hora de inicio del intervalo: "06:00-06:15" -> "06:00"
aforos2023["HORA"] = pd.to_datetime(
    aforos2023["HORARIO"].str.split("-").str[0],
    format="%H:%M"
).dt.hour

aforos2023["PERIODO"] = aforos2023["HORA"].apply(assign_period)

# agrupar por link y period, y sumar A, B, C
aforos2023 = (
    aforos2023.groupby(["LinkNo", "FromNodeNo", "ToNodeNo", "PERIODO"], as_index=False)[["A", "B", "C"]]
    .sum()
)

In [34]:
aforos2023.to_excel(os.path.join(FOLDER, "Insumos IMEPLAN", "Aforos", "Aforos por periodo.xlsx"))
aforos2023

,LinkNo,FromNodeNo,ToNodeNo,PERIODO,A,B,C
0,95755,37902,44413,AP,20.0,0.0,0.0
1,95755,37902,44413,EA,168.0,0.0,1.0
2,95755,37902,44413,LM,43.0,8.0,2.0
3,95755,37902,44413,LN,6.0,0.0,0.0
4,95755,37902,44413,PP,34.0,0.0,0.0
...,...,...,...,...,...,...,...
1470,491900,204979,134837,AP,1263.0,31.0,11.0
1471,491900,204979,134837,EA,4749.0,78.0,55.0
1472,491900,204979,134837,LM,2220.0,37.0,33.0
1473,491900,204979,134837,LN,163.0,0.0,1.0


In [38]:
aforos2023_peakperiods = aforos2023[aforos2023["PERIODO"].isin(["AP","PP"])].copy()

# pasar de formato largo a ancho
aforos_wide = (
    aforos2023_peakperiods
    .pivot_table(
        index=["LinkNo", "FromNodeNo", "ToNodeNo"],
        columns="PERIODO",
        values=["A", "B", "C"],
        aggfunc="sum",
        fill_value=0
    )
)
aforos_wide.columns = [
    f"{tipo}_{periodo}"
    for tipo, periodo in aforos_wide.columns
]
aforos_wide = aforos_wide.reset_index()
aforos_wide

,LinkNo,FromNodeNo,ToNodeNo,A_AP,A_PP,B_AP,B_PP,C_AP,C_PP
0,95755,37902,44413,20.0,34.0,0.0,0.0,0.0,0.0
1,95757,37903,37902,16.0,2.0,3.0,0.0,0.0,0.0
2,96016,38015,76091,20.0,85.0,5.0,0.0,11.0,1.0
3,96016,76091,38015,1291.0,2647.0,1.0,4.0,41.0,8.0
4,96017,38015,44453,804.0,664.0,4.0,0.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...
290,491859,204966,202316,143.0,231.0,0.0,0.0,0.0,0.0
291,491880,204973,134513,564.0,913.0,15.0,21.0,2.0,1.0
292,491892,204977,134483,749.0,791.0,22.0,24.0,8.0,9.0
293,491895,204978,134479,1035.0,1600.0,5.0,1.0,8.0,9.0


### Insert to Visum

In [33]:
#Red base GDL (con links agregados por Johan y TALA)
red_base = os.path.join(FOLDER, "Red Base GDL","RedBase Conectores y Atts", "RedBase 150826.ver")

import win32com.client
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = win32com.client.constants

In [40]:
# Read links from Visum
links_from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "FromNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")]
})

links_from_visum = links_from_visum.merge(
    aforos_wide[["FromNodeNo", "ToNodeNo", "A_AP", "A_PP", "B_AP", "B_PP", "C_AP", "C_PP"]],
    on=["FromNodeNo", "ToNodeNo"],
    how="left"
)

In [42]:
cols_aforos = ["A_AP", "A_PP", "B_AP", "B_PP", "C_AP", "C_PP"]
links_from_visum[cols_aforos] = links_from_visum[cols_aforos].fillna(0)

visum_links = Visum.Net.Links

links =  links_from_visum.copy()
links = links.reset_index(drop=True)
links.index = links.index + 1

conteos_7a8 = {
    "VOL_AUTO_AP": ("A_AP", int), 
    "VOL_BUS_AP": ("B_AP", int), 
    "VOL_CARGA_AP": ("C_AP", int), 
    "VOL_AUTO_PP": ("A_PP", int), 
    "VOL_BUS_PP": ("B_PP", int), 
    "VOL_CARGA_PP": ("C_PP", int), 
}
for visum_att, (df_col, dtype) in conteos_7a8.items():
    values = list(
        zip(
            links.index,
            links[df_col].astype(dtype)
        )
    )

    visum_links.SetMultiAttValues(visum_att, values)